# flask_post_queue.py

This notebook contains a copy of `flask_post_queue.py` as a code cell for interactive inspection and testing.

In [1]:
from datetime import datetime
import sqlite3
import os
import uuid
import subprocess
import json
import logging
from db_path import get_db_path

# Configure logging
logging.basicConfig(level=logging.DEBUG, format='%(levelname)s: %(message)s')

logging.debug("Libraries imported successfully")


DEBUG: Libraries imported successfully


### Check Queue Completion
A function that checks to see if there are more items to encode

In [2]:
def check_queue_completion(db_path):
    """
    Check whether the queue is fully processed.

    Returns True when there are NO rows in `queue` with
    `datetime_pulled IS NULL` or `datetime_encoded IS NULL`.
    Returns False if any such rows exist or on error.
    """

    try:
        conn = sqlite3.connect(db_path)
        cur = conn.cursor()

        # Count queue rows that are not yet pulled or not yet encoded
        query = """
            SELECT COUNT(*)
            FROM queue
            WHERE datetime_pulled IS NULL OR datetime_encoded IS NULL
        """
        cur.execute(query)
        unprocessed_count = cur.fetchone()[0]
        conn.close()

        # If any unprocessed rows exist, return False; otherwise True
        return unprocessed_count == 0

    except Exception as e:
        logging.debug(f"✗ Error checking queue completion: {e}")
        return False
    

In [3]:
print(check_queue_completion('boilest.db'))

False


### Get Directories
Function to get the directory paths that have been added 

 Todo:
 - [ ] Figure out better connection open/close logic 

In [4]:


def get_all_directories(db_path):
    """Return all rows from the `directories` table as a list of (guid, path)."""
    conn = None
    try:
        conn = sqlite3.connect(db_path)
        cur = conn.cursor()
        cur.execute("SELECT guid, path FROM directories")
        rows = cur.fetchall()
        conn.close()
        return rows
    except Exception as e:
        logging.debug(f"✗ Error reading directories table: {e}")
        try:
            if conn:
                conn.close()
        except:
            pass
        return []


In [5]:
dbp = 'boilest.db'
rows = get_all_directories(dbp)
if not rows:
    print('No directories found.')
else:
    print('Directories:')
    for guid, path in rows:
        print(f'{guid} -> {path}')

Directories:
1ae5df46-7d84-44ec-8174-73f1d12170c5 -> /Boil/Media/Anime
eb120fd2-5860-409d-9c47-534cefd0dc9b -> /Boil/Media/TV
1a2c2141-cc92-43a2-8c41-9ce95f0864dd -> /Boil/Media/Movies


### Directory Search
Scans for files in a given directory and yeilds the results
Todo:
- [ ] TBD

In [6]:
def find_video_files(directory_path, extensions=None):
    """Yield (directory, filename) tuples for video files under `directory_path`.

    `extensions` should be a list of extensions (with leading dot).
    If not provided a sensible default set will be used.
    """
    if extensions is None:
        extensions = ['.mp4', '.mkv', '.avi', '.mov', '.flv', '.wmv', '.ts']
    # Normalize to lowercase for comparison
    lower_exts = {e.lower() for e in extensions}

    directory_path = os.path.expanduser(directory_path)
    if not os.path.isdir(directory_path):
        logging.debug(f'Not a directory: {directory_path}')
        return

    for root, dirs, files in os.walk(directory_path):
        for fname in files:
            _, ext = os.path.splitext(fname)
            if ext.lower() in lower_exts:
                # Yield directory path (root) and filename separately
                yield root, fname



In [7]:

for d, f in find_video_files('C:/Boil/Media/Anime'):
     print(d, f)

C:/Boil/Media/Anime test_file_01.mp4
C:/Boil/Media/Anime\Media A test_file_02.mp4
C:/Boil/Media/Anime\Media A test_file_03.mp4


### FFProbe Function
Todo:
- [ ] Research expanding the entries on the ffprobe string for HDR and other criteria

In [8]:
def run_ffprobe(directory_path, filename):
    """Run ffprobe for a file.

    Backwards-compatible: if `filename` is None, `path_or_dir` is treated as a full file path.
    Otherwise `path_or_dir` is a directory and `filename` is joined to it."""
    try:
        file_path = os.path.join(directory_path, filename)

        cmd = [
            'ffprobe',
            '-loglevel', 'quiet',
            '-show_entries', 'format:stream=index,stream,codec_type,codec_name,channel_layout,format=nb_streams',  
            '-of', 'json',
            file_path
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        
        if result.returncode != 0:
            return {'error': f'ffprobe failed: {result.stderr}'}
        
        probe_data = json.loads(result.stdout)
        
        # Pretty-print the ffprobe JSON output
        logging.debug(json.dumps(probe_data, indent=2))
        
        return probe_data
    
    except FileNotFoundError:
        return {'error': 'ffprobe not found. Ensure ffmpeg is installed and in PATH.'}
    except subprocess.TimeoutExpired:
        return {'error': 'ffprobe timeout (file too large or network issue)'}
    except json.JSONDecodeError:
        return {'error': 'Invalid ffprobe JSON output'}
    except Exception as e:
        return {'error': str(e)}


### Check Codecs

Loops through the streams in stream_info from requires_encoding, then calls 
functions to determine if the steam needs encoding based on stream type conditions 

Todo:
- [x] Copy over the stream looping function from Boilest v1.0
- [ ] Research SVT-AV1 best practices for various media types
- [ ] Store SVT-AV1 best practice presets in the DB
- [ ] Call best-practive presets in check_video_stream
- [ ] Determine what audio codec to go with
- [ ] Determine what the compromises will be if ASS subtitles are re-encoded as SubRip
- [ ] Determine if there are consequences for deleting attachments 

In [9]:
def check_codecs(encoding_decision,stream_info, ffmpeg_command):
    streams_count = stream_info['format']['nb_streams']
    
    for i in range (0,streams_count):
        codec_type = stream_info['streams'][i]['codec_type'] 
        if codec_type == 'video':
            logging.debug('Stream ' + str(i) + ' is video')
            encoding_decision, ffmpeg_command = check_video_stream(encoding_decision, i, stream_info, ffmpeg_command)
        elif codec_type == 'audio':
            encoding_decision, ffmpeg_command = check_audio_stream(encoding_decision, i, stream_info, ffmpeg_command)
            logging.debug('audio stream')
        elif codec_type == 'subtitle':
            encoding_decision, ffmpeg_command = check_subtitle_stream(encoding_decision, i, stream_info, ffmpeg_command)
            logging.debug('subtitle stream')
        elif codec_type == 'attachment':
            encoding_decision, ffmpeg_command = check_attachmeent_stream(encoding_decision, i, stream_info, ffmpeg_command) 
            logging.debug('attachment stream')    
    logging.debug(encoding_decision)   
    logging.debug(ffmpeg_command)
    return encoding_decision, ffmpeg_command


In [10]:
def check_video_stream(encoding_decision, i, stream_info, ffmpeg_command):
    # Checks the video stream from check_codecs to determine if the stream needs encoding
    codec_name = stream_info['streams'][i]['codec_name'] 
    desired_video_codec = 'av1'
    logging.debug('Steam ' + str(i) + ' codec is: ' + codec_name)
    if codec_name == desired_video_codec:
        ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:v copy'
    elif codec_name == 'mjpeg':
        ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:v copy'
    elif codec_name != desired_video_codec: 
        encoding_decision = True
        svt_av1_string = "libsvtav1 -crf 25 -preset 4 -g 240 -pix_fmt yuv420p10le -svtav1-params filmgrain=20:film-grain-denoise=0:tune=0:enable-qm=1:qm-min=0:qm-max=15"
        ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:v ' + svt_av1_string
    else:
        logging.debug('ignoring for now')
    return encoding_decision, ffmpeg_command


In [11]:
def check_audio_stream(encoding_decision, i, stream_info, ffmpeg_command):
    # Checks the audio stream from check_codecs to determine if the stream needs encoding
    codec_name = stream_info['streams'][i]['codec_name'] 
    # This will be populated at a later date
    #desired_audio_codec = 'aac'
    #if codec_name != desired_video_codec:
    #    encoding_decision = True
    logging.debug('Steam ' + str(i) + ' codec is: ' + codec_name)
    ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:a copy'
    return encoding_decision, ffmpeg_command


In [12]:
def check_subtitle_stream(encoding_decision, i, stream_info, ffmpeg_command):
    # Checks the subtitle stream from check_codecs to determine if the stream needs encoding
    codec_name = stream_info['streams'][i]['codec_name'] 
    # This will be populated at a later date
    #desired_subtitle_codec = 'srt'
    #if codec_name != desired_subtitle_codec:
    #    encoding_decision = True
    logging.debug('Steam ' + str(i) + ' codec is: ' + codec_name)
    ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:s copy'
    return encoding_decision, ffmpeg_command


In [13]:
def check_attachmeent_stream(encoding_decision, i, stream_info, ffmpeg_command):
    # Checks the attachment stream from check_codecs to determine if the stream needs encoding
    # This will be populated at a later date
    #desired_attachment_codec = '???'
    #if codec_name != desired_attachment_codec:
    #    encoding_decision = True
    # Note, attachments may not have a codec name if the attachment is an image
    ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:t copy'
    return encoding_decision, ffmpeg_command


### File Output Naming Function

In [14]:
def output_file_name(file_path, encoding_decision):
    # Get the filename and current extension
    filename = os.path.basename(file_path)
    name_without_ext = os.path.splitext(filename)[0]
    current_ext = os.path.splitext(filename)[1]
    
    # Change extension to .mkv if it's not already
    if current_ext.lower() != '.mkv':
        new_filename = name_without_ext + '.mkv'
        encoding_decision = True
    else:
        new_filename = filename
    
    # Return just the new filename without any directory path
    return new_filename, encoding_decision


### Filesize Hash
Used as a quick/easy hash to validate the file is the expected.  

In [15]:
def get_file_size_kb(file_path):
    try:
        file_size_bytes = Path(file_path).stat().st_size
        file_size_kb = int(file_size_bytes / 1024)
        return file_size_kb
    except FileNotFoundError:
        logging.debug(f"✗ File not found: {file_path}")
        return 0
    except Exception as e:
        logging.debug(f"✗ Error getting file size: {e}")
        return 0

In [16]:
def is_file_unpulled_in_queue(file_name: str, directory_path: str, db_path: str) -> bool:
    """Return True if `file_name` exists in `queue` with matching `directory_path`
    and `datetime_pulled` IS NULL.

    Minimal version with no fallback logic; requires columns `input_file_name`,
    `directory_path`, and `datetime_pulled` to exist in the `queue` table.
    """
    if not file_name or not os.path.exists(db_path):
        return False

    conn = sqlite3.connect(db_path)
    try:
        cur = conn.cursor()
        cur.execute(
            "SELECT 1 FROM queue WHERE input_file_name = ? AND directory_path = ? AND datetime_pulled IS NULL LIMIT 1",
            (file_name, directory_path),
        )
        return cur.fetchone() is not None
    finally:
        conn.close()


In [17]:
print(is_file_unpulled_in_queue('test.mkv', 'C:/Boil/Media/Anime', 'boilest.db'))


False


### Write to Queue table


In [18]:
def write_to_queue(directory_guid, directory_path, input_file_name, output_file_name, before_file_size, ffmpeg_string, db_path=None):
    """Write a row into `queue` using the updated schema.

    Schema columns inserted:
      directory_guid, file_guid, directory_path, input_file_name,
      output_file_name, before_file_size, after_file_size, ffmpeg_string,
      datetime_added, datetime_pulled, datetime_encoded

    Parameters:
      - directory_guid (str)
      - directory_path (str)
      - input_file_name (str)
      - output_file_name (str)
      - before_file_size (int)
      - ffmpeg_string (str)
      - after_file_size (int|None) optional
      - db_path (str|None) optional DB path; falls back to global `db_path` variable
    """
    if db_path is None:
        db_path = 'boilest.db'

    try:
        file_guid = str(uuid.uuid4())
        datetime_added = datetime.now().isoformat()
        after_file_size = None
        datetime_pulled = None
        datetime_encoded = None

        conn = sqlite3.connect(db_path)
        cur = conn.cursor()

        cur.execute(
            "INSERT INTO queue (directory_guid, file_guid, directory_path, input_file_name, output_file_name, before_file_size, after_file_size, ffmpeg_string, datetime_added, datetime_pulled, datetime_encoded) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
            (
                directory_guid,
                file_guid,
                directory_path,
                input_file_name,
                output_file_name,
                before_file_size,
                after_file_size,
                ffmpeg_string,
                datetime_added,
                datetime_pulled,
                datetime_encoded,
            ),
        )
        conn.commit()
        conn.close()

        logging.debug(f"✓ Wrote queue entry {file_guid} for {input_file_name}")
        return file_guid

    except Exception as e:
        logging.debug(f"✗ Error writing to queue: {e}")
        try:
            conn.close()
        except Exception:
            pass
        return None


In [19]:
# Ensure write_to_queue and get_db_path (if used) are already defined/imported in this scope.
import os

# sample inputs
directory_guid = "00000000-0000-0000-0000-000000000000"
file_path = r"C:\Users\cwest\Videos\sample.mp4"   # adjust to an existing file or leave as-is
directory_path = os.path.dirname(file_path)
input_file_name = os.path.basename(file_path)
output_file_name = "sample_output.mp4"
before_file_size = os.path.getsize(file_path) if os.path.exists(file_path) else 123456
ffmpeg_string = "-c:v libx264 -preset veryfast -crf 23"

# call the function (matches signature: directory_guid, directory_path, input_file_name, output_file_name, before_file_size, ffmpeg_string)
file_guid = write_to_queue(directory_guid, directory_path, input_file_name, output_file_name, before_file_size, ffmpeg_string)

print("Inserted queue row file_guid:", file_guid)


DEBUG: ✓ Wrote queue entry 636e329f-878b-4c16-8558-a74a84ae97aa for sample.mp4


Inserted queue row file_guid: 636e329f-878b-4c16-8558-a74a84ae97aa


### Pulling it all together

In [33]:
def run_queue_workflow(db_path=None, extensions=None):

    db_path = db_path or get_db_path()
    print(db_path)

    # Only run when the queue is clear
    if not check_queue_completion(db_path):
        print('Stop: queue not empty')
        return

    directories = get_all_directories(db_path)
    total = 0
    for guid, path in directories:
        print(f"Scanning directory {path} (guid={guid})")
        for item in scan_directories_and_enqueue(path, directory_guid=guid, extensions=extensions):
            file_path = item.get('file_path') if isinstance(item, dict) else None
            if not file_path:
                print(f"Skipping item without file_path: {item}")
                continue
            try:
                print(file_path, guid)
                total += 1
            except Exception as e:
                print(f"Error processing {file_path}: {e}")

    print(f"Done. Total files processed: {total}")

run_queue_workflow()

boilest.db
Scanning directory /Boil/Media/Anime (guid=1ae5df46-7d84-44ec-8174-73f1d12170c5)


NameError: name 'scan_directories_and_enqueue' is not defined